# Day 038 Project: EDA Report on a Retail Dataset

## What You're Building

A structured EDA on a retail sales dataset: distribution analysis, top-group ranking, correlation, pivot table, and a full report dict.

**Deliverable:** You run every cell top-to-bottom. The final checks pass. You have a `report` dict and have answered three analysis questions with printed answers.

## Project Requirements

1. Load `RETAIL_CSV` (provided) into a DataFrame
2. Call `distribution_summary` on the `revenue` column
3. Call `top_groups` for product revenue — find the top 3 products
4. Call `correlation_summary` targeting `revenue`
5. Call `pivot_summary` for product × region, values=revenue, aggfunc='sum'
6. Call `eda_report` and store result as `report`
7. Verify with `_run_project_checks()`

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import io
import pandas as pd

import pandas as pd

def distribution_summary(df: pd.DataFrame, col: str) -> dict:
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        return {
            'count':      int(s.count()),
            'mean':       round(float(s.mean()), 4),
            'std':        round(float(s.std()), 4),
            'min':        float(s.min()),
            'q25':        float(s.quantile(0.25)),
            'median':     float(s.quantile(0.50)),
            'q75':        float(s.quantile(0.75)),
            'max':        float(s.max()),
            'null_count': int(s.isnull().sum()),
        }
    counts = s.value_counts()
    return {
        'count':      int(s.count()),
        'unique':     int(s.nunique()),
        'top':        str(counts.index[0]) if len(counts) else None,
        'top_freq':   int(counts.iloc[0])  if len(counts) else 0,
        'null_count': int(s.isnull().sum()),
    }


import pandas as pd

def top_groups(df: pd.DataFrame, group_col: str, value_col: str,
               n: int = 5) -> pd.DataFrame:
    return (
        df.groupby(group_col)[value_col]
        .agg(total='sum', mean='mean', count='count')
        .reset_index()
        .nlargest(n, 'total')
        .reset_index(drop=True)
    )


import pandas as pd

def correlation_summary(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    corr   = df.select_dtypes(include='number').corr()[target_col].drop(target_col)
    result = pd.DataFrame({'feature': corr.index.tolist(), 'correlation': corr.values})
    result['_abs'] = result['correlation'].abs()
    result = result.sort_values('_abs', ascending=False).drop(columns='_abs')
    return result.reset_index(drop=True)


import pandas as pd

def pivot_summary(df: pd.DataFrame, index: str, columns: str,
                  values: str, aggfunc: str = 'mean') -> pd.DataFrame:
    piv = pd.pivot_table(
        df, values=values, index=index, columns=columns,
        aggfunc=aggfunc, fill_value=0,
    )
    piv.columns.name = None
    return piv.reset_index()


import pandas as pd

def eda_report(df: pd.DataFrame) -> dict:
    num_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    return {
        'shape':           df.shape,
        'null_counts':     df.isnull().sum().to_dict(),
        'numeric_summary': df[num_cols].describe().round(2).to_dict() if num_cols else {},
        'category_counts': {col: df[col].value_counts().to_dict() for col in cat_cols},
        'correlations':    df[num_cols].corr().round(4).to_dict() if len(num_cols) > 1 else {},
    }


RETAIL_CSV = (
    'order_id,product,category,region,price,quantity\n'
    '1,Widget,Electronics,North,25.0,10\n'
    '2,Gadget,Electronics,South,150.0,3\n'
    '3,Widget,Electronics,South,25.0,5\n'
    '4,Doohickey,Accessories,East,8.0,50\n'
    '5,Gadget,Electronics,East,150.0,7\n'
    '6,Widget,Electronics,East,25.0,4\n'
    '7,Doohickey,Accessories,North,8.0,20\n'
    '8,Gadget,Electronics,North,150.0,2\n'
    '9,Widget,Electronics,West,25.0,6\n'
    '10,Doohickey,Accessories,South,8.0,15\n'
    '11,Thingamajig,Accessories,North,200.0,1\n'
    '12,Thingamajig,Accessories,East,200.0,4'
)
df = pd.read_csv(io.StringIO(RETAIL_CSV))
df['revenue'] = df['price'] * df['quantity']
print(f'Loaded {len(df)} rows × {len(df.columns)} columns')
print(df.dtypes)

## Your EDA

In [ ]:
# Step 1: Distribution of revenue
# TODO: rev_dist = distribution_summary(df, 'revenue')
# TODO: print(rev_dist)

# Step 2: Top 3 products by revenue
# TODO: top3 = top_groups(df, 'product', 'revenue', n=3)
# TODO: print('\nTop 3 products:')
# TODO: print(top3)

# Step 3: Correlations with revenue
# TODO: corr = correlation_summary(df, 'revenue')
# TODO: print('\nCorrelations with revenue:')
# TODO: print(corr)

# Step 4: Pivot table — product × region revenue
# TODO: piv = pivot_summary(df, 'product', 'region', 'revenue', 'sum')
# TODO: print('\nRevenue pivot (product × region):')
# TODO: print(piv)

# Step 5: Full EDA report
# TODO: report = eda_report(df)
# TODO: print(f'\nShape: {report["shape"]}')

## Project Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: df has revenue column
    try:
        assert 'df' in globals() and 'revenue' in df.columns, \
            "'revenue' column missing — compute df['revenue'] = df['price'] * df['quantity']"
        passed += 1; print(f'\u2705 Check 1: df loaded with revenue ({len(df)} rows)')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: top3 is defined and Gadget is the top revenue product
    try:
        assert 'top3' in globals(), 'top3 not defined'
        assert len(top3) == 3, f'top3 should have 3 rows, got {len(top3)}'
        assert top3.iloc[0]['product'] == 'Gadget', \
            f'top product should be Gadget (total=1800), got {top3.iloc[0]["product"]!r}'
        passed += 1; print(f'\u2705 Check 2: top3 correct; top product=Gadget (1800 revenue)')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: corr defined with correct structure
    try:
        assert 'corr' in globals(), 'corr not defined'
        assert list(corr.columns) == ['feature', 'correlation'], \
            f'corr columns={list(corr.columns)}'
        assert 'revenue' not in corr['feature'].values, \
            'revenue should not appear in feature column'
        passed += 1; print('\u2705 Check 3: corr has feature/correlation columns; revenue excluded')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: piv is a pivot table DataFrame
    try:
        assert 'piv' in globals(), 'piv not defined'
        assert 'product' in piv.columns, "'product' column missing from pivot"
        assert piv.isnull().sum().sum() == 0, 'pivot has NaN — use fill_value=0'
        passed += 1; print('\u2705 Check 4: pivot table has no NaN')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: report dict has all required keys
    try:
        assert 'report' in globals(), 'report not defined'
        for k in ('shape', 'null_counts', 'numeric_summary', 'category_counts', 'correlations'):
            assert k in report, f'report missing key: {k!r}'
        assert report['shape'] == df.shape, \
            f'shape mismatch: {report["shape"]} vs {df.shape}'
        passed += 1; print(f'\u2705 Check 5: report complete; shape={report["shape"]}')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Loop `distribution_summary` over every column and print a formatted table
- Try `aggfunc='count'` in `pivot_summary` to see how many orders per cell
- Use `pd.crosstab(df['product'], df['region'])` as a shortcut for counts
- Serialise `report` to JSON with `json.dumps(report, default=str, indent=2)` and save to a file — `default=str` handles tuples and non-JSON-native types
- On Day 39 you will visualise this data — keep `df` handy